# Airline Satisfaction — E5 Neural GPU Models
Runs only on Kaggle GPU. No XGBoost. Compare two residual neural networks with category embeddings and training-fold-only quantile preprocessing, each across 3 folds with early stopping (up to 100 epochs). Select architecture by developmental OOF AUC. Test frozen pretrained TabICLv2 with 65,536 labeled context rows and 8 ensemble members (fallback to 32,768 on worker failure, logged explicitly), versus E1's 16,384 rows and 2 members. Worker time limit is 90 minutes per phase. Pretrained failure is recorded; completed neural-network work remains available. A 50k selection set chooses probability blends with a three-seed LightGBM control. Reused 50k report set is read after choices, not fresh independent evaluation. Final NN fits use all rows, three seeds and median fold epochs; this differs from fold-averaged validation predictions. Selected output can keep LightGBM if neural models do not improve selection AUC; neural-only and neural-blend files are saved for comparison. No automatic submission. Public score to beat: 0.95856.

Official references: [TabICL](https://github.com/soda-inria/tabicl), [TabICL API](https://tabicl.readthedocs.io/en/latest/api.html), [PyTorch AMP](https://docs.pytorch.org/docs/stable/notes/amp_examples.html).


In [ ]:
from pathlib import Path
import os
if not Path('/kaggle/working').is_dir(): raise RuntimeError('Kaggle only')
os.environ.update(OMP_NUM_THREADS='4',OPENBLAS_NUM_THREADS='4',MKL_NUM_THREADS='4')

In [ ]:
%pip install -q tabicl==2.2.0

In [ ]:
import torch
assert torch.cuda.is_available(),'Enable Kaggle GPU'
print('Kaggle GPU',torch.cuda.get_device_name(0),'count',torch.cuda.device_count())

In [ ]:
source="from pathlib import Path\nimport os,sys,json,time,gc,random,subprocess,hashlib\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom sklearn.model_selection import train_test_split,StratifiedKFold\nfrom sklearn.preprocessing import QuantileTransformer\nfrom sklearn.metrics import roc_auc_score,log_loss\nimport joblib\n\nSEEDS=[42,2026,20261028]\ndef score(y,p):return dict(auc=float(roc_auc_score(y,p)),log_loss=float(log_loss(y,p)))\ndef seed_all(s):\n    random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.cuda.manual_seed_all(s)\ndef release():gc.collect();torch.cuda.empty_cache()\ndef preprocess(a,others):\n    cats=list(a.select_dtypes(include=['object','string']).columns);nums=[c for c in a if c not in cats]\n    levels={c:{v:i+1 for i,v in enumerate(sorted(a[c].dropna().astype(str).unique()))} for c in cats}\n    med=a[nums].median().fillna(0);qt=QuantileTransformer(n_quantiles=1000,output_distribution='normal',subsample=100000,random_state=42)\n    qt.fit(a[nums].fillna(med));state=dict(cats=cats,nums=nums,levels=levels,median=med,qt=qt)\n    def transform(d):\n        num=np.concatenate([qt.transform(d[nums].fillna(med)),d[nums].isna().to_numpy()],axis=1).astype('float32')\n        cat=np.column_stack([d[c].astype(str).map(levels[c]).fillna(0).astype('int64') for c in cats])\n        return num,cat\n    return transform(a),[transform(d) for d in others],state\nclass Block(nn.Module):\n    def __init__(self,w,drop):\n        super().__init__();self.layers=nn.Sequential(nn.LayerNorm(w),nn.Linear(w,w*2),nn.GELU(),nn.Dropout(drop),nn.Linear(w*2,w),nn.Dropout(drop))\n    def forward(self,x):return x+self.layers(x)\nclass Net(nn.Module):\n    def __init__(self,nnum,cards,width,depth,drop):\n        super().__init__();self.emb=nn.ModuleList([nn.Embedding(n,min(32,max(4,int(n**.5)*2))) for n in cards])\n        self.input=nn.Linear(nnum+sum(e.embedding_dim for e in self.emb),width)\n        self.body=nn.Sequential(*[Block(width,drop) for _ in range(depth)]);self.head=nn.Sequential(nn.LayerNorm(width),nn.GELU(),nn.Linear(width,1))\n    def forward(self,n,c):\n        z=torch.cat([n]+[e(c[:,j]) for j,e in enumerate(self.emb)],dim=1)\n        return self.head(self.body(self.input(z))).squeeze(1)\ndef predict(m,arrays):\n    m.eval();ans=[]\n    with torch.inference_mode():\n        for i in range(0,len(arrays[0]),8192):\n            n=torch.as_tensor(arrays[0][i:i+8192],device='cuda');c=torch.as_tensor(arrays[1][i:i+8192],device='cuda')\n            with torch.autocast('cuda',dtype=torch.float16):p=torch.sigmoid(m(n,c))\n            ans.append(p.float().cpu().numpy())\n    p=np.concatenate(ans);assert np.isfinite(p).all();return p\ndef train_net(a,y,state,cfg,seed,path,validation=None,epochs=100):\n    seed_all(seed);cards=[len(state['levels'][c])+1 for c in state['cats']];m=Net(a[0].shape[1],cards,**cfg).cuda()\n    opt=torch.optim.AdamW(m.parameters(),lr=.001,weight_decay=.01);scaler=torch.amp.GradScaler('cuda')\n    n=torch.as_tensor(a[0],device='cuda');c=torch.as_tensor(a[1],device='cuda');t=torch.as_tensor(y,dtype=torch.float32,device='cuda')\n    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=.5,patience=3,min_lr=1e-5) if validation else None\n    best=-1;bestepoch=0;stale=0;history=[]\n    for epoch in range(1,epochs+1):\n        m.train();order=torch.randperm(len(y),device='cuda');losses=[]\n        for batch in order.split(2048):\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast('cuda',dtype=torch.float16):loss=nn.functional.binary_cross_entropy_with_logits(m(n[batch],c[batch]),t[batch])\n            if not torch.isfinite(loss):raise RuntimeError('Nonfinite neural loss')\n            scaler.scale(loss).backward();scaler.unscale_(opt);nn.utils.clip_grad_norm_(m.parameters(),1.0);scaler.step(opt);scaler.update();losses.append(float(loss.detach()))\n        if validation:\n            auc=roc_auc_score(validation[1],predict(m,validation[0]));scheduler.step(auc);history.append(dict(epoch=epoch,auc=float(auc),loss=float(np.mean(losses))))\n            print('NN epoch',epoch,'AUC',auc,'loss',np.mean(losses),flush=True)\n            if auc>best+1e-6:best=auc;bestepoch=epoch;stale=0;torch.save(m.state_dict(),path)\n            else:stale+=1\n            if stale>=12:break\n        elif epoch==1 or epoch%10==0:print('NN full epoch',epoch,'/',epochs,'loss',np.mean(losses),flush=True)\n    if validation:m.load_state_dict(torch.load(path,weights_only=True))\n    else:torch.save(m.state_dict(),path);bestepoch=epochs\n    del n,c,t,opt,scaler;release();return m,bestepoch,history\n\ndef tab_worker(config):\n    from tabicl import TabICLClassifier\n    cfg=json.loads(Path(config).read_text());root=Path(cfg['root']);tr=pd.read_csv(root/'train.csv');x=tr.drop(columns=['id','satisfaction']);y=tr.satisfaction.astype(int).to_numpy()\n    ctx=np.load(cfg['ctx']);xp=x.iloc[np.load(cfg['rows'])] if cfg['phase']=='held' else pd.read_csv(root/'test.csv').drop(columns='id')\n    m=TabICLClassifier(n_estimators=8,n_jobs=4,device='cuda',kv_cache=True,batch_size=1,model_path=cfg['checkpoint'],allow_auto_download=False,random_state=42)\n    print('TABICL CONTEXT',len(ctx),'estimators',8,flush=True);m.fit(x.iloc[ctx],y[ctx]);pred=[];positive=list(m.classes_).index(1)\n    batchdir=Path(cfg['output']).with_suffix('');batchdir.mkdir(exist_ok=True)\n    for i in range(0,len(xp),1024):\n        f=batchdir/f'{i:07d}.npy'\n        if f.exists():p=np.load(f)\n        else:p=m.predict_proba(xp.iloc[i:i+1024])[:,positive];np.save(f,p)\n        assert np.isfinite(p).all();pred.append(p)\n        if i%20480==0:print('TABICL',cfg['phase'],i,'/',len(xp),flush=True)\n    np.save(cfg['output'],np.concatenate(pred))\ndef run_tab(root,out,pool,held,checkpoint):\n    tr=pd.read_csv(root/'train.csv');y=tr.satisfaction.astype(int).to_numpy()\n    attempts=[]\n    for size in [65536,32768]:\n        cache=out/f'tabicl_{size}';cache.mkdir(exist_ok=True)\n        try:\n            for phase,available in [('held',pool),('test',np.arange(len(y)))]:\n                ctx,_=train_test_split(available,train_size=size,stratify=y[available],random_state=77)\n                cf=cache/f'{phase}_context.npy';rf=cache/'held_rows.npy';np.save(cf,ctx);np.save(rf,held);dest=cache/f'{phase}_pred.npy'\n                if not dest.exists():\n                    config=dict(root=str(root),ctx=str(cf),rows=str(rf),phase=phase,output=str(dest),checkpoint=checkpoint)\n                    fp=cache/f'{phase}.json';fp.write_text(json.dumps(config));print('START TABICL',size,phase,flush=True)\n                    subprocess.run([sys.executable,'-u',__file__,'--tab-worker',str(fp)],check=True,timeout=5400)\n            return np.load(cache/'held_pred.npy'),np.load(cache/'test_pred.npy'),dict(context=size,estimators=8,attempts=attempts)\n        except (subprocess.CalledProcessError,subprocess.TimeoutExpired) as e:\n            attempts.append(dict(context=size,error=str(e)));print('TABICL ATTEMPT FAILED',attempts[-1],flush=True)\n    (out/'tabicl_failure.json').write_text(json.dumps(attempts,indent=2));return None,None,dict(failed=True,attempts=attempts)\n\ndef main():\n    if not Path('/kaggle/working').is_dir():raise RuntimeError('Kaggle execution only')\n    assert torch.cuda.is_available(),'Kaggle GPU required';torch.set_num_threads(4)\n    start=time.time();out=Path('/kaggle/working/experiment5');out.mkdir(exist_ok=True)\n    paths=[p.parent for p in Path('/kaggle/input').rglob('train.csv') if 'playground-series-s6e10' in str(p) and (p.parent/'test.csv').exists()];assert len(paths)==1;root=paths[0]\n    tr=pd.read_csv(root/'train.csv');te=pd.read_csv(root/'test.csv');sample=pd.read_csv(root/'sample_submission.csv');x=tr.drop(columns=['id','satisfaction']);xt=te.drop(columns=['id']);y=tr.satisfaction.astype(int).to_numpy()\n    assert np.array_equal(te.id,sample.id) and tr.id.is_unique and te.id.is_unique\n    pool,held=train_test_split(np.arange(len(y)),test_size=100000,stratify=y,random_state=42);sel,rep=train_test_split(held,test_size=.5,stratify=y[held],random_state=2026);sm=np.isin(held,sel)\n    print('HOST Kaggle; GPU',torch.cuda.get_device_name(0),'count',torch.cuda.device_count(),'DATA',tr.shape,te.shape,flush=True)\n    configs=[dict(width=256,depth=3,drop=.15),dict(width=512,depth=4,drop=.2)];search=[];foldpred={};epochcounts={}\n    folds=list(StratifiedKFold(3,shuffle=True,random_state=20261029).split(pool,y[pool]))\n    for j,cfg in enumerate(configs):\n        oof=np.zeros(len(pool));hp=np.zeros(len(held));tp=np.zeros(len(te));epochs=[]\n        for k,(a,b) in enumerate(folds):\n            ti,vi=pool[a],pool[b];print('START RESNET',j,'fold',k+1,cfg,flush=True)\n            aa,bb,state=preprocess(x.iloc[ti],[x.iloc[vi],x.iloc[held],xt]);av,ah,at=bb\n            m,ep,hist=train_net(aa,y[ti],state,cfg,42+k,out/f'nn_{j}_fold{k}.pt',validation=(av,y[vi]))\n            oof[b]=predict(m,av);hp+=predict(m,ah)/3;tp+=predict(m,at)/3;epochs.append(ep)\n            joblib.dump(state,out/f'nn_{j}_fold{k}_preprocess.joblib');(out/f'nn_{j}_fold{k}_history.json').write_text(json.dumps(hist))\n            print('DONE RESNET',j,k+1,score(y[vi],oof[b]),'epoch',ep,flush=True);del m,aa,bb,av,ah,at,state;release()\n        row=dict(index=j,config=cfg,cv=score(y[pool],oof),epochs=epochs);search.append(row);foldpred[j]=(hp,tp);epochcounts[j]=epochs\n        np.savez_compressed(out/f'nn_{j}_predictions.npz',held=hp,test=tp,oof=oof);(out/'search.json').write_text(json.dumps(search,indent=2))\n    win=max(search,key=lambda r:r['cv']['auc']);j=win['index'];cfg=win['config'];epochs=int(np.median(epochcounts[j]));hp={'resnet':foldpred[j][0]};tp={'resnet_fold':foldpred[j][1]}\n    # Control uses E3's validated LightGBM recipe. No XGBoost is used.\n    import lightgbm as lgb\n    def lgbfit(a,b,seed):\n        levels={c:sorted(a[c].dropna().unique()) for c in a.select_dtypes(include=['object','string'])};a=a.copy();b=b.copy()\n        for c,lv in levels.items():a[c]=pd.Categorical(a[c],categories=lv);b[c]=pd.Categorical(b[c],categories=lv)\n        m=lgb.LGBMClassifier(n_estimators=626,learning_rate=.035,num_leaves=63,min_child_samples=120,reg_lambda=8,colsample_bytree=.85,subsample=.8,subsample_freq=1,n_jobs=4,random_state=seed,verbosity=-1,deterministic=True,force_col_wise=True)\n        m.fit(a,y[pool] if len(a)==len(pool) else y);return m.predict_proba(b)[:,1]\n    hp['lightgbm']=np.mean([lgbfit(x.iloc[pool],x.iloc[held],s) for s in SEEDS],axis=0)\n    from huggingface_hub import hf_hub_download\n    try:\n        checkpoint=hf_hub_download(repo_id='jingang/TabICL',filename='tabicl-classifier-v2-20260212.ckpt')\n        th,tt,tabstatus=run_tab(root,out,pool,held,checkpoint)\n        if th is not None:hp['tabicl']=th;tp['tabicl']=tt\n    except Exception as e:\n        tabstatus=dict(failed=True,error=repr(e));print('TABICL UNAVAILABLE',tabstatus,flush=True)\n    cols=list(hp);candidates=[]\n    weights=[np.array([a,b,10-a-b])/10 for a in range(11) for b in range(11-a)] if len(cols)==3 else [np.array([a,10-a])/10 for a in range(11)]\n    mat=np.column_stack([hp[c] for c in cols])\n    for w in weights:candidates.append(dict(weights=w.tolist(),**score(y[held][sm],(mat@w)[sm])))\n    best=max(candidates,key=lambda r:r['auc']);w=np.array(best['weights']);neural=max([r for r in candidates if sum(v for c,v in zip(cols,r['weights']) if c!='lightgbm')>0],key=lambda r:r['auc'])\n    metrics=dict(configs=search,selected_resnet=win,tabicl=tabstatus,columns=cols,selected_weights=dict(zip(cols,w.tolist())),selection_candidates=candidates,\n        report={c:score(y[held][~sm],hp[c][~sm]) for c in cols},selected_report=score(y[held][~sm],(mat@w)[~sm]),best_neural_weights=neural['weights'],\n        note='Outer NN folds choose early stopping so CV is developmental. Recipe selected by CV, blends on separate selection partition. Report partition reused from E1-E4 and is not fresh independent evaluation. Final NN refits use median fold epochs, no validation labels.',baseline_public_auc=.95856)\n    (out/'metrics.json').write_text(json.dumps(metrics,indent=2));print('VALIDATION',json.dumps(metrics),flush=True)\n    pd.DataFrame({'id':tr.id.iloc[held].to_numpy(),'label':y[held],'partition':np.where(sm,'selection','report'),**hp}).to_csv(out/'held_predictions.csv',index=False)\n    print('FULL DATA RESNET',cfg,'epochs',epochs,flush=True);aa,bb,state=preprocess(x,[xt]);at=bb[0];npreds=[];joblib.dump(state,out/'nn_full_preprocess.joblib')\n    for seed in SEEDS:\n        m,_,_=train_net(aa,y,state,cfg,seed,out/f'nn_full_seed{seed}.pt',epochs=epochs);npreds.append(predict(m,at));del m;release()\n    tp['resnet']=np.mean(npreds,axis=0);del aa,bb,at,state;release();tp['lightgbm']=np.mean([lgbfit(x,xt,s) for s in SEEDS],axis=0)\n    testmat=np.column_stack([tp[c] for c in cols]);outputs={**tp,'selected':testmat@w,'neural_blend':testmat@np.array(neural['weights'])}\n    for name,p in outputs.items():\n        assert len(p)==len(sample) and np.isfinite(p).all() and ((p>=0)&(p<=1)).all();s=sample.copy();s.satisfaction=p;s.to_csv(out/f'submission_{name}.csv',index=False)\n    metrics['runtime_seconds']=time.time()-start;(out/'metrics.json').write_text(json.dumps(metrics,indent=2));print('COMPLETED',metrics['runtime_seconds'],flush=True)\nif __name__=='__main__':\n    if '--tab-worker' in sys.argv:tab_worker(sys.argv[sys.argv.index('--tab-worker')+1])\n    else:main()\n"
Path('/kaggle/working/run_experiment5.py').write_text(source)

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-u','/kaggle/working/run_experiment5.py'],check=True)

In [ ]:
import json,pandas as pd
from IPython.display import display
m=json.loads(Path('/kaggle/working/experiment5/metrics.json').read_text())
print(m['selected_weights'],m['tabicl'])
display(pd.DataFrame(m['report']).T)